# Import libraries

In [1]:
# !pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118
# !pip install ultralytics
# !pip install pi-heif

# !pip install timm albumentations matplotlib
# !pip install numpy

In [2]:
from pathlib import Path

from ultralytics.utils.downloads import download
from ultralytics.utils import ASSETS_URL, TQDM

# Download
dir = Path("./data")  # dataset root dir
urls = [
    f"{ASSETS_URL}/VOCtrainval_06-Nov-2007.zip",  # 446MB, 5012 images
    # f"{ASSETS_URL}/VOCtest_06-Nov-2007.zip",  # 438MB, 4953 images
    # f"{ASSETS_URL}/VOCtrainval_11-May-2012.zip",  # 1.95GB, 17126 images
]
download(urls, dir=dir, threads=3, exist_ok=True)  # download and unzip over existing (required)

Unzipping data\VOCtrainval_06-Nov-2007.zip to D:\CODE\Machine-Learning-Deep-Learning-2025\AIO2025_MeinCode\Module9\data\VOCdevkit...: 100% ━━━━━━━━━━━━ 10945/10945 2.0Kfiles/s 5.4s0.0s


In [3]:
from collections import Counter

import os
import random
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
import torchvision
from albumentations.pytorch import ToTensorV2
from termcolor import colored
from torch import nn, optim
from torch.utils.data import DataLoader, Subset, SubsetRandomSampler


d:\Utilities\miniconda\envs\cv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Reproducibility

In [4]:
def seed_everything(seed=42):
    """
    Ensure full reproducibility on Google Colab (GPU T4).
    Must be called BEFORE model/dataloader initialization.
    """

    # Python built-in RNG
    random.seed(seed)

    # NumPy RNG
    np.random.seed(seed)

    # PyTorch RNG (CPU + GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensures deterministic behavior in CUDA kernels
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    # Force deterministic algorithms
    torch.use_deterministic_algorithms(True)

    # Disable cuDNN auto-tuner
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    print(f"Seed set to {seed} (deterministic mode enabled)")

seed_everything(42)

Seed set to 42 (deterministic mode enabled)


# Put code from previous questions

In [5]:
def convert_cellboxes(predictions, S=7, C=20, B=2, anchors=None):
    predictions = predictions.to("cpu")
    batch_size = predictions.shape[0]
    predictions = predictions.reshape(batch_size, S, S, C + 5 * B)

    if anchors is None:
        anchors = torch.tensor([[1.0, 1.0]] * B, dtype=predictions.dtype)
    anchors = anchors.to(predictions.dtype)

    pred_class_logits = predictions[..., :C]
    pred_anchor = predictions[..., C:].reshape(batch_size, S, S, B, 5)

    pred_obj = torch.sigmoid(pred_anchor[..., 0])
    pred_tx_ty = torch.sigmoid(pred_anchor[..., 1:3])
    pred_tw_th = pred_anchor[..., 3:5]

    anchor_wh = anchors.view(1, 1, 1, B, 2)
    pred_wh_cell = torch.exp(pred_tw_th) * anchor_wh

    cell_x = torch.arange(S).repeat(batch_size, S, 1).unsqueeze(-1).unsqueeze(-1)
    cell_y = cell_x.permute(0, 2, 1, 3, 4)

    pred_x = (pred_tx_ty[..., 0:1] + cell_x) / S
    pred_y = (pred_tx_ty[..., 1:2] + cell_y) / S
    pred_wh = pred_wh_cell / S
    pred_box_xywh = torch.cat((pred_x, pred_y, pred_wh), dim=-1)

    # BƯỚC 1 — Xác định lớp tốt nhất cho mỗi cell
    # Convert logits to probabilities using softmax
    pred_class_probs = torch.softmax(pred_class_logits, dim=-1)  # (batch, S, S, C)

    # Find best class_id and max probability for each cell
    class_id = torch.argmax(pred_class_probs, dim=-1, keepdim=True)  # (batch, S, S, 1)
    p_class_max = torch.max(pred_class_probs, dim=-1, keepdim=True)[0]  # (batch, S, S, 1)

    # Calculate score for each anchor
    scores = pred_obj * p_class_max  # (batch, S, S, B) - broadcasts (batch, S, S, B) * (batch, S, S, 1)

    # BƯỚC 2 — Expand class for all anchors
    class_id_expanded = class_id.unsqueeze(-2).expand(batch_size, S, S, B, 1)  # (batch, S, S, B, 1)

    # BƯỚC 3 — Prepare score tensor
    scores = scores.unsqueeze(-1)  # (batch, S, S, B, 1)

    # BƯỚC 4 — Concatenate final tensor [class_id, score, b_x, b_y, b_w, b_h]
    converted_preds = torch.cat([class_id_expanded.float(), scores, pred_box_xywh], dim=-1)  # (batch, S, S, B, 6)

    return converted_preds


def cellboxes_to_boxes(out, S=7, C=20, B=2, anchors=None, threshold=0.5):
    converted_pred = convert_cellboxes(out, S=S, C=C, B=B, anchors=anchors).reshape(
        out.shape[0], S * S * B, 6
    )
    all_bboxes = []

    for ex_idx in range(out.shape[0]):
        # BƯỚC 1 — Lấy các box của ảnh hiện tại
        boxes = converted_pred[ex_idx]  # shape (S*S*B, 6)

        # BƯỚC 2 — Lọc các box có score > threshold
        boxes = boxes[boxes[:, 1] > threshold]  # shape (num_kept_boxes, 6)

        # BƯỚC 3 — Chuyển các box còn lại sang Python list
        boxes = boxes.tolist()

        # BƯỚC 4 — Ép class_id về kiểu int
        for box in boxes:
            box[0] = int(box[0])

        # BƯỚC 5 — Thêm danh sách box của ảnh hiện tại vào all_bboxes
        all_bboxes.append(boxes)

    return all_bboxes


# Custom Dataset Class

In [6]:
class CustomVOCDataset(torchvision.datasets.VOCDetection):
    def init_config_yolo(
        self,
        class_mapping,
        S=7,
        B=2,
        C=20,
        anchors=None,
        custom_transforms=None,
    ):
        # Initialize YOLO-specific configuration parameters.
        self.S = S  # Grid size S x S
        self.B = B  # Number of bounding boxes / anchors
        self.C = C  # Number of classes
        self.class_mapping = class_mapping  # Mapping of class names to class indices
        self.custom_transforms = custom_transforms
        self.anchors = torch.tensor(
            anchors if anchors is not None else [[1.0, 1.0]] * B,
            dtype=torch.float32,
        )

    def __getitem__(self, index):
        # Get an image and its target (annotations) from the VOC dataset.
        image, target = super(CustomVOCDataset, self).__getitem__(index)
        img_width, img_height = image.size

        # Convert target annotations to YOLO format bounding boxes.
        boxes = convert_to_yolo_format(
            target, img_width, img_height, self.class_mapping
        )

        just_boxes = boxes[:, 1:]
        labels = boxes[:, 0]

        # Transform
        if self.custom_transforms:
            sample = {"image": np.array(image), "bboxes": just_boxes, "labels": labels}

            sample = self.custom_transforms(**sample)
            image = sample["image"]
            boxes = sample["bboxes"]
            labels = sample["labels"]

        # Create an empty label matrix for YOLO ground truth.
        label_matrix = torch.zeros((self.S, self.S, self.C + 5 * self.B))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.float32)
        image = torch.as_tensor(image, dtype=torch.float32)

        # Iterate through each bounding box in YOLO format.
        for box, class_label in zip(boxes, labels):
            x, y, width, height = box.tolist()
            class_label = int(class_label)

            # Calculate the grid cell (i, j) that this box belongs to.
            i, j = int(self.S * y), int(self.S * x)
            i = min(self.S - 1, max(0, i))
            j = min(self.S - 1, max(0, j))
            x_cell, y_cell = self.S * x - j, self.S * y - i

            # Calculate the width and height of the box relative to the grid cell.
            width_cell, height_cell = width * self.S, height * self.S

            gt_wh = torch.tensor([width_cell, height_cell], dtype=torch.float32)
            anchor_ious = intersection_over_union(
                torch.cat((torch.zeros(2), gt_wh)).unsqueeze(0),
                torch.cat((torch.zeros((self.B, 2)), self.anchors), dim=1),
                box_format="midpoint",
            ).squeeze(-1)
            best_anchor = int(torch.argmax(anchor_ious))

            obj_idx = self.C + best_anchor * 5
            box_idx_start = obj_idx + 1

            # STEP 1 — Kiểm tra anchor đã được gán hay chưa
            if label_matrix[i, j, obj_idx] == 1:
                continue

            # STEP 2 — Gán objectness
            label_matrix[i, j, obj_idx] = 1

            # STEP 3 — Tính regression target theo anchor
            anchor_width, anchor_height = self.anchors[best_anchor].tolist()
            epsilon = 1e-6
            t_x = x_cell
            t_y = y_cell
            t_w = torch.log(torch.tensor(width_cell / (anchor_width + epsilon) + epsilon, dtype=torch.float32))
            t_h = torch.log(torch.tensor(height_cell / (anchor_height + epsilon) + epsilon, dtype=torch.float32))

            # STEP 4 — Ghi tọa độ box vào label_matrix
            label_matrix[i, j, box_idx_start : box_idx_start + 4] = torch.tensor(
                [t_x, t_y, t_w, t_h], dtype=torch.float32
            )

            # STEP 5 — Gán nhãn lớp
            label_matrix[i, j, class_label] = 1

        return image, label_matrix

# Other Utility Functions

In [7]:
def convert_to_yolo_format(target, img_width, img_height, class_mapping):
    """
    Convert annotation data from VOC format to YOLO format.

    Parameters:
    target (dict): Annotation data from VOCDetection dataset.
    img_width (int): Width of the original image.
    img_height (int): Height of the original image.
    class_mapping (dict): Mapping from class names to integer IDs.

    Returns:
    torch.Tensor: Tensor of shape [N, 5] for N bounding boxes,
                  each with [class_id, x_center, y_center, width, height].
    """
    # Extract the list of annotations from the target dictionary.
    annotations = target["annotation"]["object"]

    # Get the real width and height of the image from the annotation.
    real_width = int(target["annotation"]["size"]["width"])
    real_height = int(target["annotation"]["size"]["height"])

    # Ensure that annotations is a list, even if there's only one object.
    if not isinstance(annotations, list):
        annotations = [annotations]

    # Initialize an empty list to store the converted bounding boxes.
    boxes = []

    # Loop through each annotation and convert it to YOLO format.
    for anno in annotations:
        xmin = int(anno["bndbox"]["xmin"]) / real_width
        xmax = int(anno["bndbox"]["xmax"]) / real_width
        ymin = int(anno["bndbox"]["ymin"]) / real_height
        ymax = int(anno["bndbox"]["ymax"]) / real_height

        # Calculate the center coordinates, width, and height of the bounding box.
        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        width = xmax - xmin
        height = ymax - ymin

        # Retrieve the class name from the annotation and map it to an integer ID.
        class_name = anno["name"]
        class_id = class_mapping[class_name] if class_name in class_mapping else 0

        # Append the YOLO formatted bounding box to the list.
        boxes.append([class_id, x_center, y_center, width, height])

    # Convert the list of boxes to a torch tensor.
    return np.array(boxes)


In [8]:
def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Calculate the Intersection over Union (IoU) between bounding boxes.

    Parameters:
        boxes_preds (tensor): Predicted bounding boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): Ground truth bounding boxes (BATCH_SIZE, 4)
        box_format (str): Box format, can be "midpoint" or "corners".

    Returns:
        tensor: Intersection over Union scores for each example.
    """

    # Check if the box format is "midpoint"
    if box_format == "midpoint":
        # Calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for predicted boxes
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

        # Calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    # Check if the box format is "corners"
    if box_format == "corners":
        # Extract coordinates for predicted boxes
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]

        # Extract coordinates for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # Calculate coordinates of the intersection rectangle
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # Compute the area of the intersection rectangle, clamp(0) to handle cases where they do not overlap
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # Calculate the areas of the predicted and ground truth boxes
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # Calculate the Intersection over Union, adding a small epsilon to avoid division by zero
    return intersection / (box1_area + box2_area - intersection + 1e-6)


In [9]:
def non_max_suppression(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    Perform Non-Maximum Suppression on a list of bounding boxes.
    Parameters:
        bboxes (list): List of bounding boxes, each represented as [class_pred, prob_score, x1, y1, x2, y2].
        iou_threshold (float): IoU threshold to determine correct predicted bounding boxes.
        threshold (float): Threshold to discard predicted bounding boxes (independent of IoU).
        box_format (str): "midpoint" or "corners" to specify the format of bounding boxes.
    Returns:
        list: List of bounding boxes after performing NMS with a specific IoU threshold.
    """

    # Check the data type of the input parameter
    assert type(bboxes) == list

    # Filter predicted bounding boxes based on probability threshold
    bboxes = [box for box in bboxes if box[1] > threshold]

    # Sort bounding boxes by probability in descending order
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # List to store bounding boxes after NMS
    bboxes_after_nms = []

    # Continue looping until the list of bounding boxes is empty
    while bboxes:
        # Get the bounding box with the highest probability
        chosen_box = bboxes.pop(0)

        # Remove bounding boxes with IoU greater than the specified threshold with the chosen box
        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        # Add the chosen bounding box to the list after NMS
        bboxes_after_nms.append(chosen_box)

    # Return the list of bounding boxes after NMS
    return bboxes_after_nms


In [10]:
def mean_average_precision(
    pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint", num_classes=20
):
    """
    Calculate the mean average precision (mAP).

    Parameters:
        pred_boxes (list): A list containing predicted bounding boxes with each box defined as [train_idx, class_prediction, prob_score, x1, y1, x2, y2].
        true_boxes (list): Similar to pred_boxes but containing information about true boxes.
        iou_threshold (float): IoU threshold, where predicted boxes are considered correct.
        box_format (str): "midpoint" or "corners" used to specify the format of the boxes.
        num_classes (int): Number of classes.

    Returns:
        float: The mAP value across all classes with a specific IoU threshold.
    """

    # List to store mAP for each class
    average_precisions = []

    # Small epsilon to stabilize division
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # Iterate through all predictions and targets, and only add those belonging to
        # the current class 'c'.
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        # Find the number of boxes for each training example.
        # The Counter here counts the number of target boxes we have
        # for each training example, so if image 0 has 3, and image 1 has 5,
        # we'll have a dictionary like:
        # amount_bboxes = {0: 3, 1: 5}
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # We then loop through each key, val in this dictionary and convert it to the following (for the same example):
        # amount_bboxes = {0: torch.tensor([0, 0, 0]), 1: torch.tensor([0, 0, 0, 0, 0])}
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # Sort by box probability, index 2 is the probability
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))
        total_true_bboxes = len(ground_truths)

        # If there are no ground truth boxes for this class, it can be safely skipped
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # Only consider ground truth boxes with the same training index as the prediction
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            num_gts = len(ground_truth_img)
            best_iou = 0

            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:]),
                    box_format=box_format,
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # Only detect ground truth once
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    # True positive and mark this bounding box as seen
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1

            # If IOU is lower, the detection result is false positive
            else:
                FP[detection_idx] = 1

        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))
        # Use torch.trapz for numerical integration
        average_precisions.append(torch.trapz(precisions, recalls))

    return sum(average_precisions) / len(average_precisions)


def get_bboxes_training(
    outputs,
    labels,
    iou_threshold=0.5,
    threshold=0.4,
    box_format="midpoint",
    device="cuda",
):
    all_pred_boxes = []
    all_true_boxes = []

    # Ensure the model is in evaluation mode before obtaining bounding boxes
    train_idx = 0

    true_bboxes = cellboxes_to_boxes(labels, anchors=torch.tensor(ANCHORS, dtype=torch.float32), threshold=threshold)
    bboxes = cellboxes_to_boxes(outputs, anchors=torch.tensor(ANCHORS, dtype=torch.float32), threshold=0.0)

    for idx in range(outputs.shape[0]):
        nms_boxes = non_max_suppression(
            bboxes[idx],
            iou_threshold=iou_threshold,
            threshold=threshold,
            box_format=box_format,
        )

        for nms_box in nms_boxes:
            all_pred_boxes.append([train_idx] + nms_box)

        for box in true_bboxes[idx]:
            all_true_boxes.append([train_idx] + box)

        train_idx += 1

    return all_pred_boxes, all_true_boxes


# Function to save model and optimizer state to a checkpoint file
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)


# Model architecture

In [11]:
class SimpleYolo(nn.Module):
    def __init__(self, in_channels=3, **kwargs):
        """
        Initializes the model with ResNet34 as the backbone.

        Args:
            in_channels (int): Number of input channels. Default is 3 for RGB images.
            **kwargs: Additional keyword arguments such as split_size, num_boxes, num_classes.
        """
        super(SimpleYolo, self).__init__()
        self.split_size = kwargs.get("split_size", 7)  # S
        self.num_boxes = kwargs.get("num_boxes", 2)  # B
        self.num_classes = kwargs.get("num_classes", 20)  # C

        # Initialize ResNet34 backbone from timm
        # `features_only=True` returns a list of feature maps from specified stages
        # `out_indices=[4]` corresponds to the last layer's output
        self.backbone = timm.create_model(
            "resnet34",
            pretrained=True,
            features_only=True,
            out_indices=[4],
            in_chans=in_channels,
        )

        # Use nearest-neighbor resize instead of AdaptiveAvgPool2d so
        # torch.use_deterministic_algorithms(True) works consistently on CUDA.
        self.target_size = (self.split_size, self.split_size)

        # Fully connected layers
        self.fcs = self._create_fcs()

    @torch.autocast(device_type="cuda", dtype=torch.float32)
    def forward(self, x):
        """
        Forward pass of the YOLOv1 model.

        Args:
            x (torch.Tensor): Input tensor of shape (N, C, H, W).

        Returns:
            torch.Tensor: Output tensor containing bounding box predictions.
        """
        # Extract features using ResNet34 backbone
        features = self.backbone(x)[
            0
        ]  # timm returns a list; take the first (and only) element

        # Resize features deterministically to match spatial dimensions
        resized_features = F.interpolate(features, size=self.target_size, mode="nearest")

        # Pass through fully connected layers
        output = self.fcs(resized_features)

        return output

    def _create_fcs(self):
        """
        Creates the fully connected layers for YOLOv1.

        Returns:
            nn.Sequential: Sequential container of fully connected layers.
        """
        S, B, C = self.split_size, self.num_boxes, self.num_classes

        # Calculate the input features for the first linear layer
        # ResNet34's last feature map has 512 channels
        # After pooling to (S x S), the total features are 512 * S * S
        input_features = 512 * S * S

        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_features, 4096),
            nn.Dropout(0.0),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, S * S * (C + B * 5)),
        )


# Loss Functions

In [12]:
class YoloLoss(nn.Module):
    """
    Calculate the loss for a simple anchor-based YOLO model.
    """

    def __init__(self, S=7, B=2, C=20, anchors=None):
        super(YoloLoss, self).__init__()
        self.mse = nn.MSELoss(reduction="sum")

        self.S = S
        self.B = B
        self.C = C
        self.anchors = torch.tensor(
            anchors if anchors is not None else [[1.0, 1.0]] * B,
            dtype=torch.float32,
        )

        # YOLO-style weights
        self.lambda_noobj = 0.5
        self.lambda_coord = 5

    def forward(self, predictions, target):
        predictions = predictions.reshape(-1, self.S, self.S, self.C + self.B * 5)

        pred_class = predictions[..., : self.C]
        target_class = target[..., : self.C]

        pred_anchor = predictions[..., self.C :].reshape(-1, self.S, self.S, self.B, 5)
        target_anchor = target[..., self.C :].reshape(-1, self.S, self.S, self.B, 5)

        exists_box = target_anchor[..., 0:1]
        has_obj_cell = torch.max(exists_box, dim=3).values

        # Box loss for assigned anchors only
        pred_xy = torch.sigmoid(pred_anchor[..., 1:3])
        pred_tw_th = pred_anchor[..., 3:5]
        target_xy = target_anchor[..., 1:3]
        target_tw_th = target_anchor[..., 3:5]

        box_loss = self.mse(
            torch.flatten(exists_box * torch.cat((pred_xy, pred_tw_th), dim=-1)),
            torch.flatten(exists_box * torch.cat((target_xy, target_tw_th), dim=-1)),
        )

        # Objectness loss
        pred_obj = torch.sigmoid(pred_anchor[..., 0:1])
        target_obj = target_anchor[..., 0:1]
        object_loss = self.mse(
            torch.flatten(exists_box * pred_obj),
            torch.flatten(exists_box * target_obj),
        )

        # No-object loss
        no_object_loss = self.mse(
            torch.flatten((1 - exists_box) * pred_obj, start_dim=1),
            torch.flatten((1 - exists_box) * target_obj, start_dim=1),
        )

        # Class loss only for cells containing at least one object
        class_loss = self.mse(
            torch.flatten(has_obj_cell * pred_class, end_dim=-2),
            torch.flatten(has_obj_cell * target_class, end_dim=-2),
        )

        loss = (
            self.lambda_coord * box_loss
            + object_loss
            + self.lambda_noobj * no_object_loss
            + class_loss
        )

        return loss


# Configuration

In [13]:
# Hyperparameters and configurations
# Learning rate for the optimizer.
LEARNING_RATE = 2e-5
# Specify whether to use "cuda" (GPU) or "cpu" for training.
DEVICE = "cuda"
# Originally 64 in the research paper, but using a smaller batch size due to GPU limitations.
BATCH_SIZE = 64
# Number of training epochs.
EPOCHS = 2
# Number of worker processes for data loading.
NUM_WORKERS = 0
# If True, DataLoader will pin memory to transfer data to the GPU faster.
PIN_MEMORY = True

LOAD_MODEL_FILE = "final_yolo.pth.tar"

# Image size
WIDTH = 448
HEIGHT = 448

# Anchor sizes in cell units (width, height).
ANCHORS = [
    [1.0, 1.0],
    [2.0, 2.0],
]


# Augmentations

In [14]:
def get_train_transforms():
    return A.Compose(
        [
            A.Resize(height=HEIGHT, width=WIDTH, p=1.0),
            ToTensorV2(p=1.0),
        ],
        p=1.0,
        bbox_params=A.BboxParams(
            format="yolo",
            min_area=0,
            min_visibility=0,
            label_fields=["labels"],
        ),
    )


def get_val_transforms():
    return A.Compose(
        [
            A.Resize(height=HEIGHT, width=WIDTH, p=1.0),
            ToTensorV2(p=1.0),
        ],
        p=1.0,
        bbox_params=A.BboxParams(
            format="yolo",
            min_area=0,
            min_visibility=0,
            label_fields=["labels"],
        ),
    )


# Training Loop

In [15]:
class_mapping = {
    "aeroplane": 0,
    "bicycle": 1,
    "bird": 2,
    "boat": 3,
    "bottle": 4,
    "bus": 5,
    "car": 6,
    "cat": 7,
    "chair": 8,
    "cow": 9,
    "diningtable": 10,
    "dog": 11,
    "horse": 12,
    "motorbike": 13,
    "person": 14,
    "pottedplant": 15,
    "sheep": 16,
    "sofa": 17,
    "train": 18,
    "tvmonitor": 19,
}


In [16]:
def initialize_model():
    """Initialize the model, optimizer, and loss function."""
    model = SimpleYolo(split_size=7, num_boxes=2, num_classes=20).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = YoloLoss(S=7, B=2, C=20, anchors=ANCHORS)
    return model, optimizer, loss_fn


In [17]:
from torch.utils.data import ConcatDataset


def prepare_data():
    """Prepare datasets and create data loaders for training, validation and testing."""
    # Prepare training dataset
    train_dataset = CustomVOCDataset(
        root="./data",
        year="2007",
        image_set="train",
        download=False,
    )
    train_dataset.init_config_yolo(
        class_mapping=class_mapping,
        custom_transforms=get_train_transforms(),
        anchors=ANCHORS,
    )

    # Prepare validation dataset
    val_dataset = CustomVOCDataset(
        root="./data",
        year="2007",
        image_set="val",
        download=False,
    )
    val_dataset.init_config_yolo(
        class_mapping=class_mapping,
        custom_transforms=get_val_transforms(),
        anchors=ANCHORS,
    )

    # Combine datasets
    combined_dataset = ConcatDataset([train_dataset, val_dataset])
    total_size = len(combined_dataset)

    # Calculate split sizes (7:2:1 ratio)
    train_size = int(0.7 * total_size)
    val_size = int(0.2 * total_size)
    test_size = total_size - train_size - val_size

    # Generate indices for splits
    indices = list(range(total_size))
    rng = np.random.default_rng(42)
    rng.shuffle(indices)

    train_indices = indices[:train_size]
    val_indices = indices[train_size : train_size + val_size]
    test_indices = indices[train_size + val_size :]

    # Create samplers
    sampler_generator = torch.Generator().manual_seed(42)
    train_sampler = SubsetRandomSampler(train_indices, generator=sampler_generator)
    val_sampler = SubsetRandomSampler(val_indices, generator=sampler_generator)
    test_sampler = SubsetRandomSampler(test_indices, generator=sampler_generator)

    # Create data loaders
    train_loader = DataLoader(
        dataset=combined_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        sampler=train_sampler,
        drop_last=False,
    )

    val_loader = DataLoader(
        dataset=combined_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        sampler=val_sampler,
        drop_last=False,
    )

    test_loader = DataLoader(
        dataset=combined_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        sampler=test_sampler,
        drop_last=False,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
        combined_dataset,
        len(train_indices),
        len(val_indices),
        len(test_indices),
    )


In [18]:
def initialize_train_mAP_loader(train_dataset):
    """Initialize DataLoader for training mAP calculation."""
    map_sampler_generator = torch.Generator().manual_seed(42)
    train_mAP_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        sampler=SubsetRandomSampler(list(range(len(train_dataset))), generator=map_sampler_generator),
        drop_last=False,
    )
    return train_mAP_loader


def save_best_checkpoint(model, optimizer, filename=LOAD_MODEL_FILE):
    """Save the model checkpoint."""
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    save_checkpoint(checkpoint, filename=filename)
    print(colored(f"Checkpoint saved to {filename}", "cyan"))


In [19]:
def log_dataset_statistics(num_train, num_val, num_test):
    """Log the number of samples in each dataset split."""
    print(colored(f"Number of training samples: {num_train}", "cyan"))
    print(colored(f"Number of validation samples: {num_val}", "cyan"))
    print(colored(f"Number of test samples: {num_test}", "cyan"))


In [20]:
def train_fn(train_loader, model, optimizer, loss_fn, epoch):
    mean_loss = []
    total_batches = len(train_loader)
    display_interval = max(
        total_batches // 10, 1
    )  # Update after 10% of the total batches

    model.train()  # Ensure the model is in training mode

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = loss_fn(out, y)
        # Normalize loss by batch size
        normalized_loss = loss / x.size(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        mean_loss.append(normalized_loss.item())

        if (batch_idx + 1) % display_interval == 0 or (batch_idx + 1) == total_batches:
            print(
                f"Epoch: {epoch:3} \t Iter: {batch_idx + 1:3}/{total_batches:3} \t Loss: {normalized_loss.item():.10f}"
            )

    avg_loss = sum(mean_loss) / len(mean_loss)
    print(colored(f"Train \t loss: {avg_loss:.10f}", "green"))
    return avg_loss


def val_test_fn(data_loader, model, loss_fn, epoch, is_test=False):
    model.eval()
    mean_loss = []
    mean_mAP = []

    with torch.no_grad():  # Disable gradient calculations for val/test
        for batch_idx, (x, y) in enumerate(data_loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            out = model(x)
            loss = loss_fn(out, y)
            # Normalize loss by batch size
            normalized_loss = loss / x.size(0)

            pred_boxes, true_boxes = get_bboxes_training(
                out, y, iou_threshold=0.5, threshold=0.04
            )

            mAP = mean_average_precision(
                pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint"
            )

            mean_loss.append(normalized_loss.item())
            mean_mAP.append(mAP.item())

    avg_loss = sum(mean_loss) / len(mean_loss)
    avg_mAP = sum(mean_mAP) / len(mean_mAP)

    if is_test:
        print(
            colored(f"Test \t loss: {avg_loss:.10f} \t mAP: {avg_mAP:.10f}", "yellow")
        )
    else:
        print(colored(f"Val \t loss: {avg_loss:.10f} \t mAP: {avg_mAP:.10f}", "blue"))

    model.train()  # Put the model back in training mode
    return avg_mAP


In [21]:
def train():
    """Main training function orchestrating the training process."""
    seed_everything(42)
    # Initialize model, optimizer, and loss function
    model, optimizer, loss_fn = initialize_model()

    # Prepare datasets
    (
        train_loader,
        val_loader,
        test_loader,
        train_dataset,
        num_train,
        num_val,
        num_test,
    ) = prepare_data()

    # Log dataset statistics
    log_dataset_statistics(num_train, num_val, num_test)

    # Initialize variables to track best mAP
    best_mAP_val = 0
    best_mAP_test = 0

    # Initialize DataLoader for training mAP calculation
    train_mAP_loader = initialize_train_mAP_loader(train_dataset)

    # Training loop
    for epoch in range(1, EPOCHS + 1):
        print(colored(f"\nStarting Epoch {epoch}/{EPOCHS}", "magenta"))

        # Training for one epoch
        avg_train_loss = train_fn(train_loader, model, optimizer, loss_fn, epoch)

        # Perform Validation and Testing every 2 epochs
        if epoch % 2 == 0:
            # Validation
            print(colored(f"\nValidation at Epoch {epoch}", "magenta"))
            val_mAP = val_test_fn(val_loader, model, loss_fn, epoch, is_test=False)
            train_mAP = val_test_fn(
                train_mAP_loader, model, loss_fn, epoch, is_test=False
            )
            print(colored(f"Train mAP during Validation: {train_mAP:.10f}", "green"))
            print(colored(f"Validation mAP: {val_mAP:.10f}", "blue"))

            # Update best validation mAP
            if val_mAP > best_mAP_val:
                best_mAP_val = val_mAP
                save_best_checkpoint(model, optimizer)
                print(colored(f"New best Val mAP: {best_mAP_val:.10f}", "blue"))

            # Testing
            print(colored(f"\nTesting at Epoch {epoch}", "magenta"))
            test_mAP = val_test_fn(test_loader, model, loss_fn, epoch, is_test=True)

            # Update best test mAP
            if test_mAP > best_mAP_test:
                best_mAP_test = test_mAP
                print(colored(f"New best Test mAP: {best_mAP_test:.10f}", "yellow"))

    # Training completion message
    print(colored("\nTraining Completed!", "magenta"))
    print(colored(f"Best Val mAP: {best_mAP_val:.10f}", "blue"))
    print(colored(f"Best Test mAP: {best_mAP_test:.10f}", "yellow"))


In [22]:
train()


Seed set to 42 (deterministic mode enabled)


Number of training samples: 3507
Number of validation samples: 1002
Number of test samples: 502

Starting Epoch 1/2
Epoch:   1 	 Iter:   5/ 55 	 Loss: 27.0100727081
Epoch:   1 	 Iter:  10/ 55 	 Loss: 24.7692070007
Epoch:   1 	 Iter:  15/ 55 	 Loss: 20.7458992004
Epoch:   1 	 Iter:  20/ 55 	 Loss: 19.4552898407
Epoch:   1 	 Iter:  25/ 55 	 Loss: 21.0818672180
Epoch:   1 	 Iter:  30/ 55 	 Loss: 16.2629451752
Epoch:   1 	 Iter:  35/ 55 	 Loss: 17.0953845978
Epoch:   1 	 Iter:  40/ 55 	 Loss: 13.9285202026
Epoch:   1 	 Iter:  45/ 55 	 Loss: 15.1603393555
Epoch:   1 	 Iter:  50/ 55 	 Loss: 13.6402273178
Epoch:   1 	 Iter:  55/ 55 	 Loss: 12.9795761108
Train 	 loss: 19.3646588412

Starting Epoch 2/2
Epoch:   2 	 Iter:   5/ 55 	 Loss: 10.9725360870
Epoch:   2 	 Iter:  10/ 55 	 Loss: 8.4542961121
Epoch:   2 	 Iter:  15/ 55 	 Loss: 8.6589088440
Epoch:   2 	 Iter:  20/ 55 	 Loss: 9.1402635574
Epoch:   2 	 Iter:  25/ 55 	 Loss: 10.4095764160
Epoch:   2 	 Iter:  30/ 55 	 Loss: 9.0136089325
Epoch: 